In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

df = pd.read_csv('train-test.csv')

print("Dataset Shape:", df.shape)
print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Data Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Numeric Summary ---")
display(df.describe())



Dataset Shape: (48000, 14)

--- First 5 Rows ---


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28



--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB

--- Missing Values ---
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [2]:
df_cleaned = df.dropna(subset=['weight', 'market_index']).copy()
print(f"Rows remaining after dropping nulls: {len(df_cleaned)}")

df_cleaned['weight'] = df_cleaned['weight'].abs()


df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])

print("\n--- Missing Values ---")
print(df_cleaned.isnull().sum())

print("\n--- Numeric Summary ---")
display(df_cleaned.describe())


Rows remaining after dropping nulls: 47327

--- Missing Values ---
load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
dtype: int64

--- Numeric Summary ---


,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate
count,47327.000000,47327.000000,47327.000000,47327.000000,47327.000000,47327.000000,47327,47327.000000,47327.000000,47327.000000
mean,35.647251,-90.936758,35.641488,-90.851863,1135.573761,31425.306316,2025-05-31 20:11:02.387220736,1.083523,2.062329,2373.959449
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,5000.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25930.000000,2025-03-18 00:00:00,0.949890,1.890680,1251.650000
50%,35.294790,-88.089150,35.294790,-87.528710,952.700000,31503.000000,2025-05-31 00:00:00,1.055910,2.055600,2030.180000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.500000,37070.000000,2025-08-15 00:00:00,1.219760,2.221615,3330.695000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000
std,4.315426,13.487386,4.318321,13.467526,728.331533,7996.128819,NaN,0.168072,0.291410,1487.502747


In [3]:
# FIX: keep an unscaled copy so fill values (medians) are in the original units
df_raw = df_cleaned.copy()

features_to_normalize = ['distance', 'weight', 'market_index', 'quote_signal']

scaler = StandardScaler()
df_cleaned[features_to_normalize] = scaler.fit_transform(df_cleaned[features_to_normalize])

print("\n--- Features after Normalization ---")
display(df_cleaned[features_to_normalize].describe().round(3))


--- Features after Normalization ---


,distance,weight,market_index,quote_signal
count,47327.000,47327.000,47327.000,47327.000
mean,0.000,0.000,0.000,-0.000
std,1.000,1.000,1.000,1.000
min,-1.463,-3.305,-2.422,-4.702
25%,-0.803,-0.687,-0.795,-0.589
50%,-0.251,0.010,-0.164,-0.023
75%,0.700,0.706,0.811,0.547
max,3.164,2.010,2.286,5.312


In [4]:
df_cleaned['month'] = df_cleaned['date'].dt.month
df_cleaned['day_of_week'] = df_cleaned['date'].dt.dayofweek

df_encoded = pd.get_dummies(df_cleaned, columns=['equipment'], drop_first=True)


features_to_drop = ['load_id', 'pickup', 'delivery', 'date', 'posted_rate']
X = df_encoded.drop(columns=features_to_drop)
y = df_encoded['posted_rate']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42,
    objective='reg:squarederror'
)

print("Training XGBoost model on normalized data...")
model.fit(X_train, y_train)

predictions = model.predict(X_val)

predictions = np.maximum(predictions, 0.01)

mae = mean_absolute_error(y_val, predictions)
rmse = np.sqrt(mean_squared_error(y_val, predictions))

print(f"Validation Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Validation Root Mean Squared Error (RMSE): ${rmse:.2f}")

Training XGBoost model on normalized data...
Validation Mean Absolute Error (MAE): $126.69
Validation Root Mean Squared Error (RMSE): $624.13


In [5]:

val_df = pd.read_csv('validation.csv')
dec_df = pd.read_csv('december-chart-inputs.csv')
val_template = pd.read_csv('validation-predictions-template.csv')

print("--- Validation Data Columns ---")
print(val_df.columns.tolist())

print("\n--- December Chart Data Columns ---")
print(dec_df.columns.tolist())

print("\n--- Missing Values in Validation ---")
print(val_df.isnull().sum()[val_df.isnull().sum() > 0])

print("\n--- Missing Values in December Chart ---")
print(dec_df.isnull().sum()[dec_df.isnull().sum() > 0])

--- Validation Data Columns ---
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

--- December Chart Data Columns ---
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

--- Missing Values in Validation ---
weight          165
market_index    249
dtype: int64

--- Missing Values in December Chart ---
Series([], dtype: int64)


In [6]:

val_df['weight'] = val_df['weight'].fillna(df_raw['weight'].median()).abs()
val_df['market_index'] = val_df['market_index'].fillna(df_raw['market_index'].median())

val_df['date'] = pd.to_datetime(val_df['date'])
val_df['month'] = val_df['date'].dt.month
val_df['day_of_week'] = val_df['date'].dt.dayofweek

val_df[features_to_normalize] = scaler.transform(val_df[features_to_normalize])

val_encoded = pd.get_dummies(val_df, columns=['equipment'])
val_X = val_encoded.reindex(columns=X_train.columns, fill_value=0)

val_preds = model.predict(val_X)
val_preds = np.maximum(val_preds, 0.01)

# FIX: match predictions to the template by load_id instead of relying on row order
pred_df = pd.DataFrame({'load_id': val_df['load_id'], 'predicted_rate': val_preds})
val_out = val_template[['load_id']].merge(pred_df, on='load_id', how='left')
assert val_out['predicted_rate'].notna().all() and len(val_out) == 12000
val_out.to_csv('validation_predictions.csv', index=False)
print("Saved validation_predictions.csv")


dec_df['market_index'] = df_raw['market_index'].median()
dec_df['quote_signal'] = df_raw['quote_signal'].median()

lex_lat = df[df['pickup'] == 'Lexington']['pickup_lat'].median()
lex_lon = df[df['pickup'] == 'Lexington']['pickup_lon'].median()
fw_lat = df[df['delivery'] == 'Fort Wayne']['delivery_lat'].median()
fw_lon = df[df['delivery'] == 'Fort Wayne']['delivery_lon'].median()

dec_df['pickup_lat'] = lex_lat
dec_df['pickup_lon'] = lex_lon
dec_df['delivery_lat'] = fw_lat
dec_df['delivery_lon'] = fw_lon

dec_df['weight'] = dec_df['weight'].abs()
dec_df[features_to_normalize] = scaler.transform(dec_df[features_to_normalize])

dec_df['date'] = pd.to_datetime(dec_df['date'])
dec_df['month'] = dec_df['date'].dt.month
dec_df['day_of_week'] = dec_df['date'].dt.dayofweek

dec_encoded = pd.get_dummies(dec_df, columns=['equipment'])
dec_X = dec_encoded.reindex(columns=X_train.columns, fill_value=0)

dec_preds = model.predict(dec_X)
dec_df['predicted_rate'] = np.maximum(dec_preds, 0.01)

dec_final = dec_df[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']].copy()

dec_original = pd.read_csv('december-chart-inputs.csv')
dec_final['distance'] = dec_original['distance']
dec_final['weight'] = dec_original['weight']

dec_final.to_csv('december-chart-inputs.csv', index=False)
print("Saved december-chart-inputs.csv")

Saved validation_predictions.csv
Saved december-chart-inputs.csv
